# 01 — Vue d'ensemble de la branche `feat/data-pipeline`

**Public visé :** étudiant curieux, pas spécialiste en informatique.

**Objectif de ce notebook :** comprendre *pourquoi* cette branche existe et *ce qu'elle produit*, avant de plonger dans le code ou les données.

---

## C'est quoi un « jumelage » de villes ?

Deux villes sont **jumelées** (sister cities / twin towns) quand elles entretiennent un partenariat officiel : échanges culturels, scolaires, économiques…

Exemples : Paris ↔ Londres, Lyon ↔ …

Notre projet collecte ces liens **dans le monde entier** pour les visualiser sur une carte (branche suivante : site web).

## Que fait cette branche Git ?

La branche **`feat/data-pipeline`** ne fait **pas** encore le site web. Elle prépare **les données** :

1. Interroger **Wikidata** (base de connaissances libre, liée à Wikipédia)
2. Stocker les jumelages proprement
3. Mettre à jour automatiquement (GitHub Actions)

```
Wikidata (Internet)
       ↓
  scripts/sync_wikidata.py
       ↓
  jumelages.csv          ← source de vérité
       ↓
  graphe_jumelages.json  ← format graphe (carte réseau)
       ↓
  scripts/build_stats.py
       ↓
  stats.json, etc.       ← chiffres pour infographies
```

> **Analogie :** Wikidata = entrepôt de faits ; notre CSV = tableau Excel propre ; le JSON graphe = format pour dessiner des liens entre points.

## Les grands choix de conception (résumé)

| Choix | Pourquoi ? |
|-------|------------|
| **Wikidata** plutôt que scraper Wikipedia | Données structurées, API officielle, moins fragile |
| **CSV** comme source de vérité | Lisible dans Excel, facile à auditer |
| **Sync delta** + **full mensuel** | Peu de requêtes réseau au quotidien ; filet de sécurité |
| **Upsert** (fusion intelligente) | Pas de doublons Paris–Lyon / Lyon–Paris |
| **User-Agent avec email** | Règle d'usage de Wikidata — on s'identifie |
| **GitHub Actions** | Refresh automatique sans intervention manuelle |

Le notebook **02** détaille chaque point. Le notebook **03** montre comment explorer les fichiers produits.

## Fichiers produits (dossier `data/processed/`)

| Fichier | Rôle en une phrase |
|---------|-------------------|
| `jumelages.csv` | Liste des paires de villes jumelées |
| `graphe_jumelages.json` | Même info en `{nodes, links}` pour la carte |
| `metadata.json` | Date du dernier run, mode delta/full |
| `stats.json` | Top villes, KPIs, matrice pays↔pays |
| `city_index.json` | Index léger pour la recherche sur le site |
| `countries.json` | Liste des pays pour filtres carte |

In [ ]:
# Cellule pratique : lister les fichiers de données disponibles
from pathlib import Path

DATA = Path("..") / "data" / "processed"

if not DATA.exists():
    print("Dossier absent. Lance d'abord : python scripts/sync_wikidata.py --full")
else:
    for fichier in sorted(DATA.iterdir()):
        taille_ko = fichier.stat().st_size / 1024
        print(f"{fichier.name:25} {taille_ko:8.1f} Ko")

## Ordre de lecture recommandé

1. **01** (ce notebook) — vue d'ensemble
2. **02** — technologies, respect de Wikidata, pipeline delta/full
3. **03** — manipuler les données avec Pandas (exploration libre)
4. **04** — petit essai Wikidata limité (`sync_sample.py`)

## Pour aller plus loin (hors notebooks)

- Lire `README.md` à la racine du projet
- Parcourir `scripts/sync_wikidata.py` avec les commentaires
- Tester une requête SPARQL sur https://query.wikidata.org/